# E3-I — Ablation graphique avec contrôles topologiques renforcés

**Objectif :** vérifier si l'avantage d'une topologie opérationnelle est robuste face à des graphes de contrôle suffisamment différents. L'expérience utilise la **même architecture diagnostique à branches locale/voisine que E3-H** (et non le GCN à deux couches E3-F2A).

**Trois graphes, mêmes entrées, cibles et budget :**
1. `I_operational` : graphe opérationnel reconstruit depuis le train, identique à E3-H.
2. `I_degree_preserving` : graphe de contrôle **au même degré par nœud et même nombre d'arêtes**, choisi parmi des randomisations par échanges d'arêtes pour **minimiser le recouvrement** avec le graphe opérationnel.
3. `I_random_edges` : graphe aléatoire simple, connecté, **même nombre d'arêtes**, sans conservation des degrés ; choisi pour minimiser le recouvrement.

**Limites :** le choix du graphe de contrôle optimise la différence structurelle, pas la performance prédictive ; aucune cible de test n'intervient. Les graphes sont fixés avant les entraînements et leurs arêtes, degrés et recouvrements sont archivés. Une différence entre graphes ne prouve pas la contribution causale de Brick/IFC/SOSA.

**Protocole :** 4 horizons × 3 graphes × 3 graines = **36 entraînements** ; 200 époques maximum, patience 25, même largeur fixée depuis E3-F2A, mêmes 13 643 cibles communes par horizon. Le résultat principal est la RMSE par graine et moyenne ± écart-type. Un bootstrap apparié sera effectué ensuite, sans sélectionner le meilleur graphe sur test.


In [ ]:
from pathlib import Path
import json, random, time, platform
import numpy as np
import pandas as pd
import networkx as nx
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT=Path.cwd().resolve()
REP=ROOT/'outputs/E3_aligned_representations'
COMMON=ROOT/'outputs/E3_common_evaluation'
GCN_OLD=ROOT/'outputs/E3_F2A_GCN_200epochs'
OUT=ROOT/'outputs/E3_I_DISTINCT_GRAPH_TOPOLOGIES'
HORIZONS=(15,30,60,120)
SEEDS=(42,123,2026)
SPLITS=('train','validation','test')
NODES=7; STEPS=5; FEATURES=3
K_NEIGHBORS=2  # Top-k utilisé dans E3-F2A pour la topologie opérationnelle
MAX_EPOCHS=200; PATIENCE=25
BATCH_SIZE=256; LR=1e-3; WEIGHT_DECAY=1e-5
MODES=('I_operational','I_degree_preserving','I_random_edges')
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert REP.is_dir() and COMMON.is_dir() and GCN_OLD.is_dir()
OUT.mkdir(parents=True,exist_ok=True)
common=pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
common['timestamp']=pd.to_datetime(common.timestamp,utc=True)
assert len(common)==91763
frozen=pd.read_csv(GCN_OLD/'05_selected_configs_and_training.csv')
WIDTH={}
for h in HORIZONS:
    widths=frozen.loc[frozen.horizon_min.eq(h),'hidden'].unique()
    assert len(widths)==1,(h,widths)
    WIDTH[h]=int(widths[0])
original=json.loads((GCN_OLD/'06_protocol.json').read_text(encoding='utf-8'))
print('Device:',device,'widths:',WIDTH)


In [ ]:
def load_horizon(h):
    folder=REP/f'h{h:03d}'
    nodes=pd.read_csv(folder/'01_graph_node_index.csv').sort_values('node_index')
    assert nodes['node_index'].tolist()==list(range(7))
    devices=nodes['uuid'].astype(str).tolist()
    data={}
    for split in SPLITS:
        X=np.load(folder/f'X_graph_{split}_scaled.npy').astype(np.float32)
        Xu=np.load(folder/f'X_graph_{split}_unscaled.npy',mmap_mode='r')
        y=np.load(folder/f'y_graph_{split}.npy').astype(np.float32)
        meta=pd.read_csv(folder/f'metadata_{split}.csv')
        meta['timestamp']=pd.to_datetime(meta['timestamp'],utc=True)
        meta['target_timestamp']=pd.to_datetime(meta['target_timestamp'],utc=True)
        idx=np.load(COMMON/f'indices_h{h:03d}_{split}.npy').astype(np.int64)
        assert X.shape==(len(y),5,7,3) and y.shape[1]==7
        assert len(meta)==y.size==len(idx)*0+7*len(y)
        assert len(idx)=={'train':64393,'validation':13727,'test':13643}[split]
        assert len(set(idx.tolist()))==len(idx) and idx.min()>=0 and idx.max()<y.size
        assert meta['uuid'].astype(str).tolist()==devices*len(y), 'Unexpected node order.'
        assert meta['split'].eq(split).all()
        assert (meta['target_timestamp']-meta['timestamp']).eq(pd.Timedelta(minutes=h)).all()
        assert np.allclose(meta['target'].to_numpy(),y.reshape(-1),rtol=0,atol=1e-5)
        assert np.isfinite(X).all() and np.isfinite(y).all()
        assert np.allclose(Xu.transpose(0,2,1,3).reshape(-1,5,3)[idx],
                           np.load(folder/f'X_sequence_{split}_unscaled.npy',mmap_mode='r')[idx],
                           rtol=0,atol=0)
        selected=meta.iloc[idx][['uuid','timestamp','split']].sort_values(['uuid','timestamp','split']).reset_index(drop=True)
        expected=common.loc[common.split.eq(split),['uuid','timestamp','split']].sort_values(['uuid','timestamp','split']).reset_index(drop=True)
        assert selected.equals(expected),f'Common keys differ h={h} split={split}'
        data[split]=dict(X=X,y=y,meta=meta,idx=idx,
                         persistence=np.asarray(Xu[:,-1,:,0],dtype=np.float64).reshape(-1))
        print(h,split,X.shape,'common nodes',len(idx))
    return data,devices

def build_topologies(train_X, h):
    # Multivariate train-only similarity: flatten all time snapshots and history
    # for each variable; standardized features are already train-only scaled.
    # Pearson correlation computed separately per variable, then averaged.
    v=train_X.transpose(2,0,1,3).reshape(7,-1,3)
    similarity=np.zeros((7,7),dtype=np.float64)
    for feature in range(3):
        z=v[:,:,feature]
        corr=np.corrcoef(z)
        assert np.isfinite(corr).all(), 'Constant training trajectory; inspect topology.'
        similarity+=corr/3
    np.fill_diagonal(similarity,-np.inf)
    edges=set()
    for i in range(7):
        order=sorted((j for j in range(7) if j!=i),
                     key=lambda j:(-similarity[i,j],j))
        for j in order[:K_NEIGHBORS]:
            edges.add(tuple(sorted((i,j))))
    g=nx.Graph();g.add_nodes_from(range(7));g.add_edges_from(sorted(edges))
    # Degree-preserving null topology. Fail if no genuine rewiring possible.
    shuffled=None
    for trial in range(100):
        cand=g.copy()
        try:
            nx.double_edge_swap(cand,nswap=2,max_tries=1000,seed=42+h+trial)
        except nx.NetworkXAlgorithmError:
            continue
        if set(cand.edges())!=set(g.edges()) and sorted(dict(cand.degree()).items())==sorted(dict(g.degree()).items()):
            shuffled=cand
            break
    assert shuffled is not None,'Could not construct a distinct degree-preserving shuffled topology.'
    assert dict(g.degree())==dict(shuffled.degree())
    def Ahat(graph):
        A=nx.to_numpy_array(graph,nodelist=list(range(7)),dtype=np.float32)
        A+=np.eye(7,dtype=np.float32)
        d=A.sum(axis=1)
        return torch.tensor(A/np.sqrt(d[:,None]*d[None,:]),dtype=torch.float32,device=device)
    topologies={'G_native':nx.empty_graph(7),
                'G_operational':g,
                'G_shuffled':shuffled}
    rows=[]
    for name,graph in topologies.items():
        for i,j in graph.edges():
            rows.append({'horizon_min':h,'topology':name,'node_i':i,'node_j':j})
    pd.DataFrame(rows,columns=['horizon_min','topology','node_i','node_j']).to_csv(
        OUT/f'h{h:03d}_topology_edges.csv',index=False)
    return {k:Ahat(v) for k,v in topologies.items()}, {
        k:sorted(tuple(sorted(e)) for e in v.edges()) for k,v in topologies.items()
    }


In [ ]:
class SeparatedNeighborModel(nn.Module):
    def __init__(self,hidden,mode):
        super().__init__()
        self.mode=mode
        self.self_fc=nn.Linear(STEPS*FEATURES,hidden)
        self.neighbor_fc=nn.Linear(STEPS*FEATURES,hidden)
        self.dropout=nn.Dropout(0.1)
        self.head=nn.Linear(hidden,1)

    def forward(self,x,A_neighbors):
        assert x.ndim==4 and tuple(x.shape[1:])==(STEPS,NODES,FEATURES)
        z=x.permute(0,2,1,3).contiguous().reshape(x.shape[0],NODES,STEPS*FEATURES)
        if self.mode=='H1_local':
            h=self.self_fc(z)
        elif self.mode=='H2_neighbors':
            h=self.neighbor_fc(torch.einsum('ij,bjf->bif',A_neighbors,z))
        elif self.mode in ('H3_both','H4_both_shuffled','I_operational','I_degree_preserving','I_random_edges'):
            h=self.self_fc(z)+self.neighbor_fc(torch.einsum('ij,bjf->bif',A_neighbors,z))
        else:
            raise ValueError(self.mode)
        return self.head(self.dropout(torch.relu(h))).squeeze(-1)

def adjacency_no_self(edges):
    A=np.zeros((NODES,NODES),dtype=np.float32)
    for i,j in edges:
        A[i,j]=1;A[j,i]=1
    np.fill_diagonal(A,0)
    deg=A.sum(axis=1)
    assert np.all(deg>0),'Neighbor-only requires a neighbor for every device'
    A=A/deg[:,None]
    assert np.allclose(np.diag(A),0)
    assert np.allclose(A.sum(axis=1),1)
    return torch.tensor(A,device=device)

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass

def loader(d,split,shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(d[split]['X']),
                                    torch.from_numpy(d[split]['y'])),
                      batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=0)

@torch.no_grad()
def predict(model,d,split,A):
    model.eval()
    return np.concatenate([model(x.to(device),A).cpu().numpy()
                           for x,_ in loader(d,split)],axis=0)

def train_model(d,hidden,mode,A,seed,checkpoint):
    seed_all(seed)
    model=SeparatedNeighborModel(hidden,mode).to(device)
    # Exclude inactive branch parameters from optimizer (no inadvertent weight decay).
    if mode=='H1_local':
        params=list(model.self_fc.parameters())+list(model.head.parameters())
    elif mode=='H2_neighbors':
        params=list(model.neighbor_fc.parameters())+list(model.head.parameters())
    else:
        params=list(model.parameters())
    optimizer=torch.optim.Adam(params,lr=LR,weight_decay=WEIGHT_DECAY)
    best=np.inf;stale=0;history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();losses=[]
        for x,y in loader(d,'train',shuffle=True):
            x=x.to(device);y=y.to(device)
            optimizer.zero_grad()
            loss=nn.functional.mse_loss(model(x,A),y)
            loss.backward();optimizer.step()
            losses.append(float(loss.detach().cpu()))
        pv=predict(model,d,'validation',A)
        val_rmse=float(np.sqrt(np.mean((pv-d['validation']['y'])**2)))
        history.append({'epoch':epoch,'train_MSE':float(np.mean(losses)),
                        'validation_RMSE':val_rmse})
        if val_rmse<best-1e-7:
            best=val_rmse;stale=0
            torch.save(model.state_dict(),checkpoint)
        else:
            stale+=1
            if stale>=PATIENCE:break
    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True))
    return model,pd.DataFrame(history),best,sum(p.numel() for p in params)

def rmse(y,p):
    return float(np.sqrt(np.mean((np.asarray(y)-np.asarray(p))**2)))

# Structural non-leakage tests: change target's own history or other devices.
for mode in MODES:
    seed_all(42)
    A=adjacency_no_self([(i,(i+1)%NODES) for i in range(NODES)])
    net=SeparatedNeighborModel(16,mode).to(device).eval()
    x=torch.randn(2,STEPS,NODES,FEATURES,device=device)
    with torch.no_grad():
        base=net(x,A)
        altered_self=x.clone();altered_self[:,:,0,:]+=50
        changed_self=net(altered_self,A)
        altered_other=x.clone();altered_other[:,:,1:,:]+=50
        changed_other=net(altered_other,A)
    if mode=='H1_local':
        assert torch.allclose(base[:,0],changed_other[:,0],atol=1e-6)
    if mode=='H2_neighbors':
        assert torch.allclose(base[:,0],changed_self[:,0],atol=1e-6)
    print('No-leakage check passed:',mode)


## Construction des contrôles topologiques

Les randomisations utilisent **uniquement les arêtes du graphe opérationnel et des graines fixes**. Le graphe préservant les degrés peut avoir un recouvrement minimal non nul du fait de la contrainte de degré et du petit nombre de nœuds (7). Nous n'imposons donc pas un seuil arbitraire impossible à atteindre : nous rapportons le **recouvrement réellement obtenu** et l'interprétons explicitement.

Le graphe aléatoire conserve le nombre d'arêtes et la connexité, mais non les degrés ; il permet un contrôle complémentaire. Aucun choix n'est effectué en fonction de la RMSE.


In [ ]:
def canonical_edges(edges):
    return tuple(sorted(tuple(sorted((int(a),int(b)))) for a,b in edges))

def make_graph(edges):
    g=nx.Graph()
    g.add_nodes_from(range(NODES))
    g.add_edges_from(canonical_edges(edges))
    assert g.number_of_nodes()==NODES and nx.is_connected(g)
    return g

def distinct_topologies(op_edges,h,attempts=1200):
    op=make_graph(op_edges)
    ref=set(canonical_edges(op.edges()))
    n=len(ref)
    # Degree-preserving double-edge swaps. Each candidate has the exact same
    # per-node degree sequence; maximize symmetric difference.
    best=None;best_overlap=n+1
    for trial in range(attempts):
        g=op.copy()
        seed=100000*h+trial
        try:
            nx.double_edge_swap(g,nswap=10+trial%20,max_tries=1000,seed=seed)
        except nx.NetworkXAlgorithmError:
            continue
        if not nx.is_connected(g):continue
        candidate=set(canonical_edges(g.edges()))
        overlap=len(ref&candidate)
        if overlap<best_overlap:
            best=g.copy();best_overlap=overlap
            if overlap==0:break
    assert best is not None,'Could not generate degree-preserving connected control'
    assert [op.degree(i) for i in range(NODES)]==[best.degree(i) for i in range(NODES)]
    # Random edge control: exactly n edges, all nodes have a neighbor, connected.
    random_best=None;random_overlap=n+1
    for trial in range(attempts):
        g=nx.gnm_random_graph(NODES,n,seed=200000*h+trial)
        if not nx.is_connected(g):continue
        overlap=len(ref&set(canonical_edges(g.edges())))
        if overlap<random_overlap:
            random_best=g.copy();random_overlap=overlap
            if overlap==0:break
    assert random_best is not None
    assert random_best.number_of_edges()==n
    return {
      'I_operational':canonical_edges(op.edges()),
      'I_degree_preserving':canonical_edges(best.edges()),
      'I_random_edges':canonical_edges(random_best.edges())
    }

# Preflight: verify topology controls before any training.
graph_edges={};topology_diagnostics=[]
for h in HORIZONS:
    d,_=load_horizon(h)
    _,built=build_topologies(d['train']['X'],h)
    ref_edges=canonical_edges(built['G_operational'])
    old_edges=canonical_edges(original['topologies'][str(h)]['G_operational'])
    assert ref_edges==old_edges,(h,'operational topology changed')
    graph_edges[h]=distinct_topologies(ref_edges,h)
    for model,edges in graph_edges[h].items():
        g=make_graph(edges)
        overlap=len(set(edges)&set(ref_edges))
        topology_diagnostics.append({
          'horizon_min':h,'graph':model,'edges':len(edges),
          'overlap_edges':overlap,'overlap_fraction':overlap/len(ref_edges),
          'degree_sequence_by_node':json.dumps([g.degree(i) for i in range(NODES)]),
          'connected':nx.is_connected(g)
        })
    print(h,'min:',[(k,len(set(v)&set(ref_edges))) for k,v in graph_edges[h].items()],
          'overlapping edges out of',len(ref_edges))
topology_df=pd.DataFrame(topology_diagnostics)
topology_df.to_csv(OUT/'00_PREFLIGHT_topology_distances.csv',index=False)
assert topology_df.groupby('horizon_min').size().eq(3).all()


## Entraînement et sauvegarde

La même architecture à deux branches est utilisée dans les trois conditions. Les trois graines contrôlent l'initialisation ; chaque condition est entraînée indépendamment, depuis zéro, avec les mêmes paramètres d'optimisation. La validation seule détermine le meilleur checkpoint.


In [ ]:
records=[];predictions=[];trainings=[];edge_records=[]
for h in HORIZONS:
    print(f'\\n===== Horizon {h} min =====')
    d,devices=load_horizon(h)
    for model_name,edges in graph_edges[h].items():
        A=adjacency_no_self(edges)
        for i,j in edges:
            edge_records.append({'horizon_min':h,'topology':model_name,'node_i':i,'node_j':j})
        for seed in SEEDS:
            t0=time.perf_counter()
            ckpt=OUT/f'h{h:03d}_{model_name}_seed{seed}.pt'
            model,history,best,nparams=train_model(d,WIDTH[h],model_name,A,seed,ckpt)
            history.to_csv(OUT/f'h{h:03d}_{model_name}_seed{seed}_history.csv',index=False)
            trainings.append({'horizon_min':h,'model':model_name,'seed':seed,
                              'hidden':WIDTH[h],'active_parameters':nparams,
                              'best_validation_RMSE':best,
                              'best_epoch':int(history.loc[history.validation_RMSE.idxmin(),'epoch']),
                              'epochs_run':len(history),
                              'training_seconds':time.perf_counter()-t0})
            idx=d['test']['idx']
            y=d['test']['y'].reshape(-1)
            p=predict(model,d,'test',A).reshape(-1)
            meta=d['test']['meta'].iloc[idx].copy().reset_index(drop=True)
            assert len(meta)==len(idx)==13643
            pred=meta[['uuid','timestamp']].copy()
            pred['horizon_min']=h;pred['model']=model_name;pred['seed']=seed
            pred['y_true']=y[idx];pred['y_pred']=p[idx]
            pred['persistence_pred']=d['test']['persistence'][idx]
            predictions.append(pred)
            records.append({'horizon_min':h,'model':model_name,'seed':seed,
                            'n':len(idx),'RMSE':rmse(y[idx],p[idx]),
                            'Persistence_RMSE':rmse(y[idx],d['test']['persistence'][idx])})
            print('Finished',h,model_name,seed,'RMSE',round(records[-1]['RMSE'],5))
            del model
            if torch.cuda.is_available():torch.cuda.empty_cache()

pred_df=pd.concat(predictions,ignore_index=True)
metric_df=pd.DataFrame(records)
train_df=pd.DataFrame(trainings)
edge_df=pd.DataFrame(edge_records)
assert len(train_df)==36
assert len(metric_df)==36
assert len(pred_df)==36*13643
assert not pred_df.duplicated(['horizon_min','model','seed','uuid','timestamp']).any()
summary=(metric_df.groupby(['horizon_min','model'],as_index=False)
         .agg(n_seeds=('seed','nunique'),RMSE_mean=('RMSE','mean'),RMSE_SD=('RMSE','std'),
              Persistence_RMSE=('Persistence_RMSE','mean')))
assert summary.n_seeds.eq(3).all()
summary.to_csv(OUT/'01_PRIMARY_I_mean_sd.csv',index=False)
metric_df.to_csv(OUT/'02_I_per_seed_metrics.csv',index=False)
pred_df.to_csv(OUT/'03_I_common_test_predictions.csv',index=False)
train_df.to_csv(OUT/'04_I_training_diagnostics.csv',index=False)
edge_df.to_csv(OUT/'05_I_edges.csv',index=False)
protocol={
 'experiment':'E3-I distinct graph topology controls',
 'architecture':'E3-H separated self and one-hop neighbor branches',
 'note':'not E3-F2A two-layer GCN',
 'graph_types':list(MODES),'topology_control':'degree preserving, and edge-count preserving connected random',
 'topology_choice':'minimum overlap with operational among pre-generated candidates, without using target/test',
 'topology_candidate_attempts':1200,
 'horizons_minutes':list(HORIZONS),'seeds':list(SEEDS),
 'max_epochs':MAX_EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE,
 'learning_rate':LR,'weight_decay':WEIGHT_DECAY,
 'test_samples_per_horizon_seed_mode':13643,
 'limitations':['degree-preserving graphs may have unavoidable edge overlap',
                'random-edge control does not preserve degrees',
                'one building and one test period, three seeds',
                'correlation-derived operational graph not a causal semantic ablation'],
 'python':platform.python_version(),'torch':torch.__version__,'networkx':nx.__version__}
(OUT/'06_I_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print(summary.to_string(index=False))
print('Saved:',OUT)


## Résultats à transmettre et interprétation

Transmettre le dossier `outputs/E3_I_DISTINCT_GRAPH_TOPOLOGIES/` compressé, ou au minimum :
- `00_PREFLIGHT_topology_distances.csv` (degrés et recouvrement),
- `01_PRIMARY_I_mean_sd.csv` (RMSE moyennes et écarts-types),
- `02_I_per_seed_metrics.csv`,
- `03_I_common_test_predictions.csv`,
- `04_I_training_diagnostics.csv`,
- `05_I_edges.csv`,
- `06_I_protocol.json`.

**Important :** les deux contrôles répondent à des questions distinctes. Le contrôle à degrés identiques est le plus strict sur les degrés ; le contrôle aléatoire explore une structure plus différente au prix d'un changement de degré. Une comparaison statistique appariée par blocs sur les mêmes prédictions sera la prochaine étape, sans sélectionner le contrôle donnant le résultat le plus favorable.
